### GPU Set up

In [77]:
import torch

if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using Apple GPU:", device)

Using Apple GPU: mps


### 1. Load & Prepare Text

In [78]:
END_TOKEN = "<END>"
input_file = "./input_data/TinyStories-valid.txt"

with open(input_file, "r", encoding="utf-8") as f:
    raw_text = f.read()


lines = [
    line.replace("@-@", "-").replace("@,@", ",").replace("@.@", ".").strip().lower()
    for line in raw_text.splitlines()
    if line.strip()
]

lines = [line for line in lines if not (line.startswith("=") and line.endswith("="))]

lines = [f"{line} {END_TOKEN}" for line in lines]

### 2. Tokenize Text

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.trainers import BpeTrainer

tokenizer = Tokenizer(BPE(unk_token="<UNK>"))

tokenizer.pre_tokenizer = Whitespace()

trainer = BpeTrainer(
    vocab_size=20000,
    special_tokens=[
        "<UNK>",
        "<|endoftext|>",
    ],
)

tokenizer.train_from_iterator(
    lines,
    trainer,
)

tokenizer.save("./output_model/tokenizer.json")

vocab_size = tokenizer.get_vocab_size()

print("vocab_size:", vocab_size)




vocab_size: 14478


### 3. Create Training Sequences

In [80]:
sequence_length = 16
max_sequences = 500_000

training_sequences = []

for line in lines:
    token_ids = tokenizer.encode(line).ids

    if len(token_ids) <= sequence_length:
        continue

    for i in range(len(token_ids) - sequence_length):
        input_ids = token_ids[i : i + sequence_length]
        target_id = token_ids[i + sequence_length]

        training_sequences.append((input_ids, target_id))

        if len(training_sequences) >= max_sequences:
            break

    if len(training_sequences) >= max_sequences:
        break


print("training_sequences: ", training_sequences[0])
print("input: ", tokenizer.decode(training_sequences[0][0]))
print("output: ", tokenizer.decode([training_sequences[0][1]]))


training_sequences:  ([573, 14, 573, 199, 75, 667, 221, 79, 129, 12, 3, 980, 12, 1602, 12, 397], 221)
input:  spot . spot saw the shiny car and said , " wow , kitty , your
output:  car


### 4. Weights & Tables

In [81]:
import torch.nn as nn

embedding_dimension = 64
attention_dimension = 64
feed_forward_dimension = embedding_dimension * 4

learning_rate = 0.001
epochs = 20
batch_size = 64

n_heads = 4
head_dimension = attention_dimension // n_heads


## tables

embedding_table = nn.Parameter(
    torch.randn(vocab_size, embedding_dimension, device=device) * 0.01
)

position_embedding_table = nn.Parameter(
    torch.randn(sequence_length, embedding_dimension, device=device) * 0.01
)

## Attention Weights

W_Q = nn.Parameter(
    torch.randn(embedding_dimension, attention_dimension, device=device) * 0.01
)
W_K = nn.Parameter(
    torch.randn(embedding_dimension, attention_dimension, device=device) * 0.01
)
W_V = nn.Parameter(
    torch.randn(embedding_dimension, attention_dimension, device=device) * 0.01
)

# Output projection after attention
W_O = nn.Parameter(
    torch.randn(attention_dimension, embedding_dimension, device=device) * 0.01
)

# LayerNorm weights
norm1_gamma = nn.Parameter(torch.ones(embedding_dimension, device=device))
norm1_beta = nn.Parameter(torch.zeros(embedding_dimension, device=device))

norm2_gamma = nn.Parameter(torch.ones(embedding_dimension, device=device))
norm2_beta = nn.Parameter(torch.zeros(embedding_dimension, device=device))


def layer_norm(x, gamma, beta, eps=1e-5):
    mean = x.mean(dim=-1, keepdim=True)
    std = x.std(dim=-1, keepdim=True)
    return gamma * ((x - mean) / (std + eps)) + beta


# Feed Forward Network weights
W1 = nn.Parameter(
    torch.randn(embedding_dimension, feed_forward_dimension, device=device) * 0.01
)
b1 = nn.Parameter(torch.zeros(feed_forward_dimension, device=device))

W2 = nn.Parameter(
    torch.randn(feed_forward_dimension, embedding_dimension, device=device) * 0.01
)
b2 = nn.Parameter(torch.zeros(embedding_dimension, device=device))

# LM head
lm_head = nn.Parameter(
    torch.randn(embedding_dimension, vocab_size, device=device) * 0.01
)

parameters = [
    embedding_table,
    position_embedding_table,
    W_Q,
    W_K,
    W_V,
    W_O,
    norm1_gamma,
    norm1_beta,
    norm2_gamma,
    norm2_beta,
    W1,
    b1,
    W2,
    b2,
    lm_head,
]

optimizer = torch.optim.Adam(parameters, lr=learning_rate)

### 5. Training Loop

In [82]:
import random
from tqdm import tqdm
import torch.nn.functional as F

for epoch in range(epochs):
    total_loss = 0
    random.shuffle(training_sequences)

    for step in tqdm(
        range(0, len(training_sequences), batch_size),
        desc=f"Epoch {epoch + 1}/{epochs}",
        unit="batch",
    ):
        # input ids -> target ids
        batch = training_sequences[step : step + batch_size]

        batch_input_ids = torch.tensor(
            [x[0] for x in batch],
            device=device,
        )

        batch_target_ids = torch.tensor(
            [x[1] for x in batch],
            device=device,
        )

        current_batch_size = batch_input_ids.shape[0]

        # Lookup input_id's embeddings
        X = embedding_table[batch_input_ids]

        # Lookup position embeddings
        positions = torch.arange(sequence_length, device=device)
        positions = positions.unsqueeze(0)

        X = X + position_embedding_table[positions]

        # normalization | Layernorm 1
        X_norm = layer_norm(
            X,
            norm1_gamma,
            norm1_beta,
        )

        # self-attention
        Q = X_norm @ W_Q
        K = X_norm @ W_K
        V = X_norm @ W_V

        Q = Q.reshape(
            current_batch_size,
            sequence_length,
            n_heads,
            head_dimension,
        ).transpose(1, 2)

        K = K.reshape(
            current_batch_size,
            sequence_length,
            n_heads,
            head_dimension,
        ).transpose(1, 2)

        V = V.reshape(
            current_batch_size,
            sequence_length,
            n_heads,
            head_dimension,
        ).transpose(1, 2)

        # attention scores
        scores = Q @ K.transpose(-2, -1)
        scores = scores / torch.sqrt(
            torch.tensor(
                head_dimension,
                device=device,
            )
        )

        # causal masking
        mask = torch.triu(
            torch.ones(
                sequence_length,
                sequence_length,
                device=device,
            ),
            diagonal=1,
        )

        scores = scores.masked_fill(
            mask == 1,
            float("-inf"),
        )

        # attention softmax
        attention_weights = F.softmax(
            scores,
            dim=-1,
        )

        # context mixing
        attention_output = attention_weights @ V

        attention_output = attention_output.transpose(1, 2).reshape(
            current_batch_size,
            sequence_length,
            attention_dimension,
        )

        # output projection
        attention_output = attention_output @ W_O

        # residual connection 1
        X = X + attention_output

        # layernorm 2
        X_norm = layer_norm(
            X,
            norm2_gamma,
            norm2_beta,
        )

        # feed forward network
        hidden = X_norm @ W1 + b1
        hidden = F.gelu(hidden)
        ffn_output = hidden @ W2 + b2

        # residual connection 2
        X = X + ffn_output

        # lm head -> logits
        last_token_vector = X[:, -1]
        logits = last_token_vector @ lm_head

        # loss

        loss = F.cross_entropy(
            logits,
            batch_target_ids,
        )

        # backprop + weight update
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

        # live progress
        progress_bar.set_postfix(loss=f"{loss.item():.4f}")

    avg_loss = total_loss / (len(training_sequences) / batch_size)
    print(f"\nEPOCH {epoch + 1}/{epochs} DONE | avg loss = {avg_loss:.4f}")

Epoch 1/20: 100%|██████████| 7813/7813 [00:46<00:00, 167.36batch/s, loss=3.6120]



EPOCH 1/20 DONE | avg loss = 4.1830


Epoch 2/20: 100%|██████████| 7813/7813 [00:45<00:00, 170.56batch/s, loss=3.8765]



EPOCH 2/20 DONE | avg loss = 3.4719


Epoch 3/20: 100%|██████████| 7813/7813 [00:41<00:00, 186.33batch/s, loss=3.0944]


...


EPOCH 18/20 DONE | avg loss = 2.3241


Epoch 19/20: 100%|██████████| 7813/7813 [00:45<00:00, 171.35batch/s, loss=2.3575]



EPOCH 19/20 DONE | avg loss = 2.3050


Epoch 20/20: 100%|██████████| 7813/7813 [00:47<00:00, 163.21batch/s, loss=2.3480]


EPOCH 20/20 DONE | avg loss = 2.2858


### 6. Save Model

In [84]:
import numpy as np

output_file = "./output_model/tiny_stories_gpt.npz"

np.savez(
    output_file,
    n_heads=n_heads,
    head_dimension=head_dimension,
    embedding_table=embedding_table.detach().cpu().numpy(),
    position_embedding_table=position_embedding_table.detach().cpu().numpy(),
    W_Q=W_Q.detach().cpu().numpy(),
    W_K=W_K.detach().cpu().numpy(),
    W_V=W_V.detach().cpu().numpy(),
    W_O=W_O.detach().cpu().numpy(),
    norm1_gamma=norm1_gamma.detach().cpu().numpy(),
    norm1_beta=norm1_beta.detach().cpu().numpy(),
    norm2_gamma=norm2_gamma.detach().cpu().numpy(),
    norm2_beta=norm2_beta.detach().cpu().numpy(),
    W1=W1.detach().cpu().numpy(),
    b1=b1.detach().cpu().numpy(),
    W2=W2.detach().cpu().numpy(),
    b2=b2.detach().cpu().numpy(),
    lm_head=lm_head.detach().cpu().numpy(),
    vocab_size=vocab_size,
    sequence_length=sequence_length,
    embedding_dimension=embedding_dimension,
    attention_dimension=attention_dimension,
    feed_forward_dimension=feed_forward_dimension,
)

print(f"Saved model to {output_file}")

Saved model to ./output_model/tiny_stories_gpt.npz
